# fNIRS Processing: Last-5s Cognitive Load + MNE-NIRS GLM Validation

Fixes: 

- uses the verified trigger-window mapping: `3 = visual blank/baseline`, `2 = game/activity`, `4 = block boundary`
- reconstructs paired onset/offset windows from `*_lsl.tri` instead of treating every annotation as a trial onset
- preprocesses the full recording before ROI summaries
- uses participant age and wavelength-specific DPF for Beer-Lambert conversion
- computes the exact last-5-second cognitive-load feature
- runs an MNE-NIRS GLM validation with HRF-convolved Activity/Baseline regressors

Main outputs are written to `fnirs_notebook_results`.

In [ ]:
from __future__ import annotations

import math
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    source_detector_distances,
    temporal_derivative_distribution_repair,
)
from mne_nirs.statistics import run_glm
from nilearn.glm.first_level import make_first_level_design_matrix

print("MNE:", mne.__version__)
try:
    import mne_nirs
    print("MNE-NIRS:", mne_nirs.__version__)
except Exception as exc:
    print("Could not read MNE-NIRS version:", exc)

In [ ]:
PROJECT_DIR = Path("/Users/colleencipriano/SurfaceA")
DATA_DIR = PROJECT_DIR / "fnirsdata"
AGES_CSV = DATA_DIR / "participant_ages.csv"
OUT_DIR = PROJECT_DIR / "fnirs_notebook_results"
OUT_DIR.mkdir(exist_ok=True)
PRIMARY_DIR = OUT_DIR / "PRIMARY"
QC_DIR = OUT_DIR / "QC"
SENSITIVITY_DIR = OUT_DIR / "SENSITIVITY"
GLM_DIR = OUT_DIR / "GLM_VALIDATION"
for directory in [PRIMARY_DIR, QC_DIR, SENSITIVITY_DIR, GLM_DIR]:
    directory.mkdir(exist_ok=True)

SCI_PRIMARY = 0.5
SCI_SENSITIVITY = 0.3
FILTER_LOW = 0.01
FILTER_HIGH = 0.09

print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

## Helper Functions

The trigger files contain paired markers. A valid baseline/game trial pair is:

```text
code 3 start -> code 3 end
code 2 start -> code 2 end
```

P13 has one extra unmatched `code 2`; P23 has one unmatched `code 2`, leaving 23 valid pairs.

In [ ]:
@dataclass(frozen=True)
class Window:
    code: int
    start_sample: int
    end_sample: int
    start_line: int | None = None
    end_line: int | None = None

    @property
    def duration_samples(self) -> int:
        return self.end_sample - self.start_sample


def participant_number(folder: Path) -> int:
    match = re.search(r"P(\d+)_fnirs", folder.name, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f"Cannot parse participant number from {folder}")
    return int(match.group(1))


def find_single(folder: Path, pattern: str) -> Path:
    matches = sorted(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern} in {folder}")
    if len(matches) > 1:
        raise ValueError(f"Expected one {pattern} in {folder}, found {len(matches)}")
    return matches[0]


def load_ages(path: Path) -> dict[int, float]:
    df = pd.read_csv(path)
    lower_cols = {str(c).strip().lower(): c for c in df.columns}
    if {"participant", "age"}.issubset(lower_cols):
        data = df[[lower_cols["participant"], lower_cols["age"]]].copy()
        data.columns = ["participant", "age"]
    else:
        data = pd.read_csv(path, header=None, names=["participant", "age"])
    data["participant"] = pd.to_numeric(data["participant"], errors="coerce")
    data["age"] = pd.to_numeric(data["age"], errors="coerce")
    data = data.dropna(subset=["participant", "age"])
    return {int(row.participant): float(row.age) for row in data.itertuples()}


def load_trigger_windows(trigger_path: Path) -> list[Window]:
    rows = []
    for line_no, line in enumerate(trigger_path.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        _, sample, code = line.split(";")
        rows.append((line_no, int(sample), int(code)))

    windows = []
    i = 0
    while i < len(rows) - 1:
        line_a, sample_a, code_a = rows[i]
        line_b, sample_b, code_b = rows[i + 1]
        if code_a == code_b and sample_b > sample_a:
            windows.append(Window(code_a, sample_a, sample_b, line_a, line_b))
            i += 2
        else:
            i += 1
    return windows


def pair_trials_with_baselines(windows: list[Window]) -> list[tuple[Window, Window]]:
    pairs = []
    previous_baseline = None
    for window in sorted(windows, key=lambda w: w.start_sample):
        if window.code == 3:
            previous_baseline = window
        elif window.code == 2 and previous_baseline is not None:
            pairs.append((window, previous_baseline))
            previous_baseline = None
    return pairs


def unmatched_code2_windows(windows: list[Window]) -> list[Window]:
    unmatched = []
    previous_baseline = None
    for window in sorted(windows, key=lambda w: w.start_sample):
        if window.code == 3:
            previous_baseline = window
        elif window.code == 2:
            if previous_baseline is None:
                unmatched.append(window)
            previous_baseline = None
    return unmatched


def dpf_scholkmann_wolf(wavelength_nm: float, age_years: float) -> float:
    # Scholkmann & Wolf (2013), age/wavelength dependent DPF.
    alpha = 223.3
    beta = 0.05624
    gamma = 0.8493
    delta = -5.723e-7
    epsilon = 0.001245
    zeta = -0.9025
    return (
        alpha
        + beta * (age_years**gamma)
        + delta * (wavelength_nm**3)
        + epsilon * (wavelength_nm**2)
        + zeta * wavelength_nm
    )


def roi_from_source_detector(source: int, detector: int) -> str | None:
    if source in range(1, 5) and detector in range(1, 5):
        return "frontal"
    if source in range(5, 9) and detector in range(5, 9):
        return "posterior"
    return None


def hbdiff_pairs_by_region(raw_hb: mne.io.BaseRaw) -> dict[str, list[tuple[int, int]]]:
    by_name = {}
    for idx, ch in enumerate(raw_hb.ch_names):
        base, chroma = ch.rsplit(" ", 1)
        by_name.setdefault(base, {})[chroma] = idx

    regions = {"frontal": [], "posterior": []}
    for base, picks in by_name.items():
        if "hbo" not in picks or "hbr" not in picks:
            continue
        source, detector = base.replace("S", "").split("_D")
        region = roi_from_source_detector(int(source), int(detector))
        if region is not None:
            regions[region].append((picks["hbo"], picks["hbr"]))
    return regions


ages = load_ages(AGES_CSV)
participant_folders = sorted(DATA_DIR.glob("P*_fnirs"), key=participant_number)
print("Participants:", [participant_number(f) for f in participant_folders])
print("Ages loaded:", len(ages))

## Verify Trigger Mapping Across Participants

In [ ]:
trigger_rows = []
for folder in participant_folders:
    pid = participant_number(folder)
    windows = load_trigger_windows(find_single(folder, "*_lsl.tri"))
    pairs = pair_trials_with_baselines(windows)
    unmatched = unmatched_code2_windows(windows)
    trigger_rows.append(
        {
            "participant": pid,
            "code2_windows": sum(w.code == 2 for w in windows),
            "code3_windows": sum(w.code == 3 for w in windows),
            "code4_windows": sum(w.code == 4 for w in windows),
            "valid_baseline_game_pairs": len(pairs),
            "unmatched_code2_windows": len(unmatched),
            "unmatched_code2_lines": "; ".join(
                f"{w.start_line}-{w.end_line}" for w in unmatched
            ),
            "first_window_codes": " ".join(str(w.code) for w in windows[:12]),
        }
    )

trigger_check = pd.DataFrame(trigger_rows)
trigger_check.to_csv(QC_DIR / "trigger_mapping_check.csv", index=False)
trigger_check

## MNE Preprocessing

In [ ]:
def build_activity_baseline_annotations(folder: Path, sfreq: float) -> mne.Annotations:
    windows = load_trigger_windows(find_single(folder, "*_lsl.tri"))
    onsets, durations, descriptions = [], [], []
    for window in windows:
        if window.code not in (2, 3):
            continue
        onsets.append(window.start_sample / sfreq)
        durations.append(window.duration_samples / sfreq)
        descriptions.append("Activity" if window.code == 2 else "Baseline")
    return mne.Annotations(onsets, durations, descriptions, orig_time=None)


def preprocess_raw_hb(folder: Path, age: float, sci_threshold: float = 0.5) -> tuple[mne.io.BaseRaw, dict]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        raw = mne.io.read_raw_nirx(folder, preload=True, verbose=False)

    picks = mne.pick_types(raw.info, meg=False, fnirs=True)
    distances = source_detector_distances(raw.info, picks=picks)
    raw.pick(picks[distances > 0.01])

    raw_od = optical_density(raw)
    sci = scalp_coupling_index(raw_od)
    bads = list(np.array(raw_od.ch_names)[sci < sci_threshold])
    raw_od.info["bads"] = bads

    raw_od = temporal_derivative_distribution_repair(raw_od)

    ppf = (dpf_scholkmann_wolf(760.0, age), dpf_scholkmann_wolf(850.0, age))
    raw_hb = beer_lambert_law(raw_od, ppf=ppf)
    if raw_hb.info["bads"]:
        raw_hb.drop_channels(raw_hb.info["bads"])

    raw_hb.filter(FILTER_LOW, FILTER_HIGH, verbose=False)
    raw_hb.set_annotations(build_activity_baseline_annotations(folder, raw.info["sfreq"]))

    diagnostics = {
        "participant": participant_number(folder),
        "age": age,
        "sci_threshold": sci_threshold,
        "n_bad_wavelength_channels": len(bads),
        "n_hb_channels_after_drop": len(raw_hb.ch_names),
        "dpf_760": ppf[0],
        "dpf_850": ppf[1],
    }
    return raw_hb, diagnostics

## Last-5s Cognitive Load Feature

For each valid trial pair:

```text
CL trial value = mean_last5_game(HbO - HbR) - mean_last5_baseline(HbO - HbR)
```

Then trial values are averaged by participant and ROI.

In [ ]:
def tail_mean_hbdiff(
    raw_hb: mne.io.BaseRaw,
    window: Window,
    channel_pairs: list[tuple[int, int]],
    seconds: float = 5.0,
) -> float:
    if not channel_pairs:
        return math.nan
    data = raw_hb.get_data()
    sfreq = raw_hb.info["sfreq"]
    end = min(data.shape[1], window.end_sample)
    n_samples = max(1, int(round(seconds * sfreq)))
    start = max(0, window.start_sample, end - n_samples)
    if start >= end:
        return math.nan
    values = []
    for hbo_idx, hbr_idx in channel_pairs:
        values.append(data[hbo_idx, start:end] - data[hbr_idx, start:end])
    return float(np.nanmean(values) * 1e6)


def compute_last5_features_for_participant(
    folder: Path,
    age: float,
    sci_threshold: float = 0.5,
) -> tuple[pd.DataFrame, dict]:
    raw_hb, diag = preprocess_raw_hb(folder, age, sci_threshold=sci_threshold)
    region_pairs = hbdiff_pairs_by_region(raw_hb)
    trial_pairs = pair_trials_with_baselines(load_trigger_windows(find_single(folder, "*_lsl.tri")))
    rows = []
    pid = participant_number(folder)
    for trial_number, (game, baseline) in enumerate(trial_pairs[:24], start=1):
        for region, pairs in region_pairs.items():
            baseline_last5 = tail_mean_hbdiff(raw_hb, baseline, pairs)
            game_last5 = tail_mean_hbdiff(raw_hb, game, pairs)
            rows.append(
                {
                    "participant": pid,
                    "age": age,
                    "trial": trial_number,
                    "region": region,
                    "n_channel_pairs": len(pairs),
                    "baseline_start_sample": baseline.start_sample,
                    "baseline_end_sample": baseline.end_sample,
                    "game_start_sample": game.start_sample,
                    "game_end_sample": game.end_sample,
                    "hbdiff_uM_last5_baseline": baseline_last5,
                    "hbdiff_uM_last5_game": game_last5,
                    "hbdiff_uM_last5_change": game_last5 - baseline_last5,
                }
            )
    return pd.DataFrame(rows), diag


def summarize_last5(
    trial_df: pd.DataFrame,
    summary_dir: Path,
    stats_dir: Path,
    label: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary = (
        trial_df.groupby(["participant", "region"], as_index=False)
        .agg(
            age=("age", "first"),
            trials=("trial", "nunique"),
            mean_channel_pairs=("n_channel_pairs", "mean"),
            hbdiff_uM_last5_mean=("hbdiff_uM_last5_change", "mean"),
            hbdiff_uM_last5_sd=("hbdiff_uM_last5_change", "std"),
        )
        .sort_values(["participant", "region"])
    )
    wide = summary.pivot_table(
        index="participant",
        columns="region",
        values="hbdiff_uM_last5_mean",
        aggfunc="first",
    ).reset_index()
    wide = wide.rename(
        columns={
            "frontal": "CL_E_DLPFC_ch1_7_hbdiff_uM",
            "posterior": "CL_V_occipital_ch8_15_hbdiff_uM",
        }
    )
    summary.to_csv(summary_dir / f"last5_roi_summary_{label}.csv", index=False)
    wide.to_csv(stats_dir / f"last5_cognitive_load_for_stats_{label}.csv", index=False)
    return summary, wide

In [ ]:
all_last5 = []
all_last5_diag = []
for folder in participant_folders:
    pid = participant_number(folder)
    df, diag = compute_last5_features_for_participant(folder, ages[pid], sci_threshold=SCI_PRIMARY)
    all_last5.append(df)
    all_last5_diag.append(diag)
    print(f"Last-5s processed P{pid}")

last5_trial_df = pd.concat(all_last5, ignore_index=True)
last5_trial_df.to_csv(QC_DIR / "last5_trial_level_sci05.csv", index=False)
pd.DataFrame(all_last5_diag).to_csv(QC_DIR / "last5_processing_diagnostics_sci05.csv", index=False)
last5_summary, last5_for_stats = summarize_last5(last5_trial_df, QC_DIR, PRIMARY_DIR, "sci05")
last5_for_stats

## Optional SCI >= 0.3 Sensitivity Run

In [ ]:
RUN_SCI03_SENSITIVITY = True

if RUN_SCI03_SENSITIVITY:
    all_last5_03 = []
    all_last5_diag_03 = []
    for folder in participant_folders:
        pid = participant_number(folder)
        df, diag = compute_last5_features_for_participant(folder, ages[pid], sci_threshold=SCI_SENSITIVITY)
        all_last5_03.append(df)
        all_last5_diag_03.append(diag)
        print(f"SCI 0.3 last-5s processed P{pid}")

    last5_trial_df_03 = pd.concat(all_last5_03, ignore_index=True)
    last5_trial_df_03.to_csv(SENSITIVITY_DIR / "last5_trial_level_sci03.csv", index=False)
    pd.DataFrame(all_last5_diag_03).to_csv(SENSITIVITY_DIR / "last5_processing_diagnostics_sci03.csv", index=False)
    last5_summary_03, last5_for_stats_03 = summarize_last5(last5_trial_df_03, SENSITIVITY_DIR, SENSITIVITY_DIR, "sci03")
    display(last5_for_stats_03.head())

## MNE-NIRS GLM Validation

This validates the feature extraction using HRF-convolved `Activity` and `Baseline` regressors.

The GLM output is not the same as the last-5s feature. It estimates a modeled `Activity - Baseline` beta contrast across the full window.

In [ ]:
def run_glm_for_participant(folder: Path, age: float, sci_threshold: float = 0.5) -> tuple[pd.DataFrame, dict]:
    raw_hb, diag = preprocess_raw_hb(folder, age, sci_threshold=sci_threshold)
    annotations = raw_hb.annotations
    events = pd.DataFrame(
        {
            "trial_type": annotations.description,
            "onset": annotations.onset - raw_hb.first_time,
            "duration": annotations.duration,
        }
    )
    design = make_first_level_design_matrix(
        raw_hb.times,
        events,
        hrf_model="glover",
        drift_model="cosine",
        high_pass=0.01,
    )
    glm = run_glm(raw_hb, design, noise_model="ar1", bins=0, n_jobs=1, verbose=0)
    df = glm.to_dataframe()
    df["participant"] = participant_number(folder)
    df["age"] = age
    df["sci_threshold"] = sci_threshold
    df["region"] = [
        roi_from_source_detector(int(row.Source), int(row.Detector)) for row in df.itertuples()
    ]
    df = df[df["region"].notna()].copy()
    df["theta_uM"] = df["theta"] * 1e6
    df["se_uM"] = df["se"] * 1e6
    return df, diag


def compute_glm_roi_contrasts(channel_df: pd.DataFrame) -> pd.DataFrame:
    task = channel_df[channel_df["Condition"].isin(["Activity", "Baseline"])].copy()
    idx = [
        "participant", "age", "sci_threshold", "region",
        "Source", "Detector", "Chroma", "ch_name",
    ]
    wide = task.pivot_table(index=idx, columns="Condition", values="theta_uM", aggfunc="first").reset_index()
    wide["activity_minus_baseline_uM"] = wide["Activity"] - wide["Baseline"]

    chroma_summary = (
        wide.groupby(["participant", "age", "sci_threshold", "region", "Chroma"], as_index=False)
        .agg(
            n_channels=("activity_minus_baseline_uM", "size"),
            glm_activity_minus_baseline_uM=("activity_minus_baseline_uM", "mean"),
        )
    )
    hbo = chroma_summary[chroma_summary["Chroma"] == "hbo"].rename(
        columns={"glm_activity_minus_baseline_uM": "glm_hbo_activity_minus_baseline_uM"}
    )
    hbr = chroma_summary[chroma_summary["Chroma"] == "hbr"].rename(
        columns={"glm_activity_minus_baseline_uM": "glm_hbr_activity_minus_baseline_uM"}
    )
    merged = hbo.merge(
        hbr[["participant", "region", "glm_hbr_activity_minus_baseline_uM"]],
        on=["participant", "region"],
        how="outer",
    )
    merged["glm_hbdiff_activity_minus_baseline_uM"] = (
        merged["glm_hbo_activity_minus_baseline_uM"]
        - merged["glm_hbr_activity_minus_baseline_uM"]
    )
    return merged


def summarize_glm_for_stats(roi_df: pd.DataFrame, out_dir: Path, label: str) -> pd.DataFrame:
    wide = roi_df.pivot_table(
        index="participant",
        columns="region",
        values="glm_hbdiff_activity_minus_baseline_uM",
        aggfunc="first",
    ).reset_index()
    wide = wide.rename(
        columns={
            "frontal": "GLM_CL_E_DLPFC_ch1_7_hbdiff_uM",
            "posterior": "GLM_CL_V_occipital_ch8_15_hbdiff_uM",
        }
    )
    wide.to_csv(out_dir / f"glm_cognitive_load_for_stats_{label}.csv", index=False)
    return wide

In [ ]:
all_glm_channels = []
all_glm_diag = []
failures = []
for folder in participant_folders:
    pid = participant_number(folder)
    try:
        df, diag = run_glm_for_participant(folder, ages[pid], sci_threshold=SCI_PRIMARY)
        all_glm_channels.append(df)
        all_glm_diag.append(diag)
        print(f"GLM processed P{pid}")
    except Exception as exc:
        failures.append({"participant": pid, "folder": folder.name, "error": str(exc)})
        print(f"GLM failed P{pid}: {exc}")

glm_channel_df = pd.concat(all_glm_channels, ignore_index=True)
glm_channel_df.to_csv(GLM_DIR / "glm_channel_coefficients_sci05.csv", index=False)
pd.DataFrame(all_glm_diag).to_csv(GLM_DIR / "glm_processing_diagnostics_sci05.csv", index=False)
pd.DataFrame(failures).to_csv(GLM_DIR / "glm_failures_sci05.csv", index=False)

glm_roi = compute_glm_roi_contrasts(glm_channel_df)
glm_roi.to_csv(GLM_DIR / "glm_roi_activity_minus_baseline_sci05.csv", index=False)
glm_for_stats = summarize_glm_for_stats(glm_roi, GLM_DIR, "sci05")
glm_for_stats

## Compare Last-5s Feature and GLM Validation

In [ ]:
comparison = last5_for_stats.merge(glm_for_stats, on="participant", how="outer")
comparison.to_csv(GLM_DIR / "last5_vs_glm_cognitive_load_comparison_sci05.csv", index=False)

print("Correlations between last-5s and GLM values:")
for a, b in [
    ("CL_E_DLPFC_ch1_7_hbdiff_uM", "GLM_CL_E_DLPFC_ch1_7_hbdiff_uM"),
    ("CL_V_occipital_ch8_15_hbdiff_uM", "GLM_CL_V_occipital_ch8_15_hbdiff_uM"),
]:
    sub = comparison[[a, b]].dropna()
    print(a, "vs", b, "n =", len(sub), "r =", sub[a].corr(sub[b]))

comparison

## Exploratory Mixed Model for GLM ROI Difference

In [ ]:
model_df = glm_roi.dropna(subset=["glm_hbdiff_activity_minus_baseline_uM"]).copy()
model_df["region"] = pd.Categorical(model_df["region"], categories=["frontal", "posterior"])

if model_df["participant"].nunique() >= 3 and model_df["region"].nunique() == 2:
    model = smf.mixedlm(
        "glm_hbdiff_activity_minus_baseline_uM ~ region",
        data=model_df,
        groups=model_df["participant"],
    )
    result = model.fit(reml=False, method="lbfgs")
    print(result.summary())
    (GLM_DIR / "mixed_effects_region_model_sci05.txt").write_text(result.summary().as_text())
else:
    print("Not enough data for mixed model.")

## Reports

Primary values:

- `PRIMARY/last5_cognitive_load_for_stats_sci05.csv`
- columns:
  - `CL_E_DLPFC_ch1_7_hbdiff_uM`
  - `CL_V_occipital_ch8_15_hbdiff_uM`

GLM validation values:

- `GLM_VALIDATION/glm_cognitive_load_for_stats_sci05.csv`
- columns:
  - `GLM_CL_E_DLPFC_ch1_7_hbdiff_uM`
  - `GLM_CL_V_occipital_ch8_15_hbdiff_uM`

`SCI >= 0.3` output only as a sensitivity check, not the primary result.